# Coffee First Crack Detection — Quickstart

This notebook demonstrates how to use the **[`syamaner/coffee-first-crack-detection`](https://huggingface.co/syamaner/coffee-first-crack-detection)** model to detect the **first crack** moment during coffee roasting.

## What you'll learn
1. Load the fine-tuned Audio Spectrogram Transformer (AST) from HuggingFace Hub
2. Classify a single 10-second audio window as `first_crack` / `no_first_crack`
3. Run sliding-window inference over a simulated roast recording
4. Visualise the probability timeline and detect the first crack onset

> **Runs on Google Colab** — no local audio files needed.  All audio comes from [`syamaner/coffee-first-crack-audio`](https://huggingface.co/datasets/syamaner/coffee-first-crack-audio).

In [ ]:
# Uncomment the line below when running in Google Colab
# !pip install -q transformers datasets librosa soundfile torch torchaudio numpy matplotlib huggingface-hub

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import ASTForAudioClassification, ASTFeatureExtractor
from datasets import load_dataset

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Auto-detect best available device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Using device: {DEVICE}")

## 1. Load Model from HuggingFace Hub

We load the fine-tuned `ASTForAudioClassification` and its `ASTFeatureExtractor` directly from the Hub.  The model was fine-tuned from [`MIT/ast-finetuned-audioset-10-10-0.4593`](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593) (~86 M parameters) for binary classification.

In [ ]:
MODEL_ID = "syamaner/coffee-first-crack-detection"

print(f"Loading model: {MODEL_ID} ...")
model = ASTForAudioClassification.from_pretrained(MODEL_ID)
extractor = ASTFeatureExtractor.from_pretrained(MODEL_ID)
model = model.to(DEVICE)
model.eval()

print(f"Labels : {model.config.id2label}")
print(f"Params : {sum(p.numel() for p in model.parameters()):,}")

## 2. Load Sample Audio from HuggingFace Dataset

[`syamaner/coffee-first-crack-audio`](https://huggingface.co/datasets/syamaner/coffee-first-crack-audio) contains 10-second, 16 kHz mono chunks labelled `first_crack` or `no_first_crack`.

We load the **test split** and separate samples by class for use in the next two sections.

In [ ]:
DATASET_ID  = "syamaner/coffee-first-crack-audio"
SAMPLE_RATE = 16_000

print(f"Loading dataset: {DATASET_ID}  (test split) ...")
ds = load_dataset(DATASET_ID, split="test")

no_crack_samples = [row for row in ds if row["label"] == "no_first_crack"]
crack_samples    = [row for row in ds if row["label"] == "first_crack"]

print(f"Test set      : {len(ds)} samples")
print(f"  no_first_crack : {len(no_crack_samples)}")
print(f"  first_crack    : {len(crack_samples)}")


def extract_audio(row: dict) -> np.ndarray:
    """Return float32 16 kHz mono array from a dataset row."""
    audio_col = row["audio"]
    arr = np.array(audio_col["array"], dtype=np.float32)
    sr  = audio_col["sampling_rate"]
    if sr != SAMPLE_RATE:
        import librosa
        arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)
    return arr

## 3. Single-Window Classification

Classify one 10-second chunk from each class and compare the probabilities.

In [ ]:
def classify_window(audio: np.ndarray, true_label: str = "") -> dict:
    """Run inference on a single audio window and return result dict."""
    inputs = extractor(audio.tolist(), sampling_rate=SAMPLE_RATE, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits
        probs  = torch.softmax(logits, dim=-1)[0].cpu().numpy()

    pred_id    = int(probs.argmax())
    pred_label = model.config.id2label[str(pred_id)]
    correct    = (pred_label == true_label) if true_label else None

    tick = " ✓" if correct is True else (" ✗" if correct is False else "")
    print(
        f"[{true_label or 'sample'}]  predicted={pred_label}  "
        f"p(no_crack)={probs[0]:.3f}  p(first_crack)={probs[1]:.3f}{tick}"
    )
    return {
        "true_label":    true_label,
        "predicted":     pred_label,
        "correct":       correct,
        "p_no_crack":    float(probs[0]),
        "p_first_crack": float(probs[1]),
    }


sample_no_crack = extract_audio(no_crack_samples[0])
sample_crack    = extract_audio(crack_samples[0])

r_no_crack = classify_window(sample_no_crack, "no_first_crack")
r_crack    = classify_window(sample_crack,    "first_crack")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

CLASS_NAMES = ["no_first_crack", "first_crack"]
COLORS      = ["#4CAF50", "#F44336"]

for ax, result, title in zip(
    axes,
    [r_no_crack, r_crack],
    ["True label: no_first_crack", "True label: first_crack"],
):
    values = [result["p_no_crack"], result["p_first_crack"]]
    bars   = ax.bar(CLASS_NAMES, values, color=COLORS, width=0.5, edgecolor="white")
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Probability")
    ax.set_title(title)
    ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2, val + 0.03, f"{val:.3f}",
            ha="center", va="bottom", fontsize=11, fontweight="bold",
        )

    pred  = result["predicted"]
    color = "#2196F3" if result["correct"] else "#E91E63"
    ax.set_xlabel(f"Prediction: {pred}", fontweight="bold", color=color)

fig.suptitle("Single-Window Classification Probabilities", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Sliding-Window Inference on a Simulated Roast Recording

We assemble a demo recording by concatenating:
- **5 × `no_first_crack`** chunks → ~50 s of background roast noise
- **5 × `first_crack`** chunks → ~50 s with cracking sounds

Then we run a sliding-window detector (10 s window, 70 % overlap = 3 s hop) and plot the probability timeline.  
A **detection event** is confirmed once ≥ 5 positive windows appear within a 20-second span.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
WINDOW_SIZE    = 10.0   # seconds
OVERLAP        = 0.7    # 70 % → 3 s hop
THRESHOLD      = 0.6    # positive classification threshold
MIN_POPS       = 5      # pops required to confirm
CONFIRM_WINDOW = 20.0   # confirmation span (seconds)

window_samples = int(WINDOW_SIZE * SAMPLE_RATE)
hop_samples    = int(window_samples * (1 - OVERLAP))

# ── Assemble demo recording ───────────────────────────────────────────────────
n_no = min(5, len(no_crack_samples))
n_cr = min(5, len(crack_samples))

no_audio  = np.concatenate([extract_audio(no_crack_samples[i]) for i in range(n_no)])
cr_audio  = np.concatenate([extract_audio(crack_samples[i])    for i in range(n_cr)])
recording = np.concatenate([no_audio, cr_audio])

transition_sec = len(no_audio) / SAMPLE_RATE
total_sec      = len(recording) / SAMPLE_RATE
print(f"Demo recording : {total_sec:.1f}s  (transition at {transition_sec:.0f}s)")

# ── Sliding window ────────────────────────────────────────────────────────────
timeline:       list[dict]                        = []
history:        list[tuple[float, bool, float]]   = []
detected_event: dict | None                       = None
start = 0

while start + window_samples <= len(recording):
    window  = recording[start : start + window_samples]
    t_start = start / SAMPLE_RATE

    # Energy-based noise gate: skip silent windows
    if np.sqrt(np.mean(window ** 2)) < 0.01:
        timeline.append({"time": t_start, "prob": 0.0})
        start += hop_samples
        continue

    inputs = extractor(window.tolist(), sampling_rate=SAMPLE_RATE, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits
        probs  = torch.softmax(logits, dim=-1)[0].cpu().numpy()

    prob_crack  = float(probs[1])
    is_positive = prob_crack >= THRESHOLD
    history.append((t_start, is_positive, prob_crack))
    timeline.append({"time": t_start, "prob": prob_crack})

    # Confirmation check
    if detected_event is None:
        cutoff = t_start - CONFIRM_WINDOW
        recent = sum(1 for t, pos, _ in history if t >= cutoff and pos)
        if recent >= MIN_POPS:
            first_t = next(t for t, pos, _ in history if pos and t >= cutoff)
            detected_event = {"time": first_t, "confidence": recent}
            print(f"FIRST CRACK DETECTED at {first_t:.1f}s  (confidence: {recent} pops)")

    start += hop_samples

if detected_event is None:
    print("No first crack detected in demo recording.")

In [ ]:
times = [r["time"] for r in timeline]
probs = [r["prob"] for r in timeline]

fig, ax = plt.subplots(figsize=(14, 5))

# Background region shading
ax.axvspan(0,              transition_sec, alpha=0.07, color="#4CAF50", label="no_first_crack region")
ax.axvspan(transition_sec, max(times),     alpha=0.07, color="#F44336", label="first_crack region")

# Threshold reference
ax.axhline(THRESHOLD, color="orange", linestyle="--", linewidth=1.2,
           label=f"Threshold ({THRESHOLD})")

# Probability line
ax.plot(times, probs, color="#607D8B", linewidth=1.2, alpha=0.6, zorder=2)

# Colour-coded scatter: below / above threshold
below = [(t, p) for t, p in zip(times, probs) if p < THRESHOLD]
above = [(t, p) for t, p in zip(times, probs) if p >= THRESHOLD]

if below:
    bx, by = zip(*below)
    ax.scatter(bx, by, color="#4CAF50", s=25, zorder=3, label="p < threshold")
if above:
    ax_, ay = zip(*above)
    ax.scatter(ax_, ay, color="#F44336", s=45, marker="^", zorder=3, label="p ≥ threshold")

# Detection event marker
if detected_event:
    det_t = detected_event["time"]
    ax.axvline(det_t, color="#E91E63", linewidth=2.0, linestyle="-",
               label=f"First crack detected @ {det_t:.1f}s")
    ax.annotate(
        f"First crack\n@ {det_t:.1f}s",
        xy=(det_t, THRESHOLD),
        xytext=(det_t + 2, THRESHOLD + 0.22),
        fontsize=10, color="#E91E63", fontweight="bold",
        arrowprops=dict(arrowstyle="->", color="#E91E63", lw=1.5),
    )

ax.set_xlabel("Time (seconds)", fontsize=12)
ax.set_ylabel("P(first_crack)", fontsize=12)
ax.set_title("Sliding Window Inference — First Crack Probability Timeline", fontsize=13)
ax.set_ylim(-0.05, 1.15)
ax.legend(loc="upper left", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Next Steps

| What | Where |
|------|-------|
| Live microphone detection | [`inference.py`](https://github.com/syamaner/coffee-first-crack-detection/blob/main/src/coffee_first_crack/inference.py) — `FirstCrackDetector(use_microphone=True)` |
| ONNX export (Raspberry Pi 5) | [`export_onnx.py`](https://github.com/syamaner/coffee-first-crack-detection/blob/main/src/coffee_first_crack/export_onnx.py) — INT8 quantized, <500 ms/window |
| Retrain on your own data | [`train.py`](https://github.com/syamaner/coffee-first-crack-detection/blob/main/src/coffee_first_crack/train.py) — HF Trainer with class-weighted loss |
| Model card | [`syamaner/coffee-first-crack-detection`](https://huggingface.co/syamaner/coffee-first-crack-detection) |
| Dataset | [`syamaner/coffee-first-crack-audio`](https://huggingface.co/datasets/syamaner/coffee-first-crack-audio) |
| Source code | [`github.com/syamaner/coffee-first-crack-detection`](https://github.com/syamaner/coffee-first-crack-detection) |